In [43]:
# zarr>=3 es obligatorio: los arreglos usan la especificación Zarr v3 (zarr.json, chunks en c/i/j/k/l)
!pip install -q --upgrade "zarr>=3" kaggle

In [44]:
import os, json, time, shutil, zipfile, tempfile
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("zarr", zarr.__version__, "| pandas", pd.__version__)

zarr 3.3.0 | pandas 2.2.3


In [45]:
# --- Autenticación de Kaggle -------------------------------------------------
# Kaggle tiene dos esquemas vigentes:
#   nuevo   -> un solo token KGAT_... en KAGGLE_API_TOKEN o ~/.kaggle/access_token
#   antiguo -> kaggle.json con username + key (KAGGLE_USERNAME / KAGGLE_KEY)
# En Colab: icono de llave (Secrets) -> crear el secreto y activar "Notebook access".

def setup_kaggle_auth():
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    def secret(name):
        v = os.environ.get(name)
        if v:
            return v
        if userdata is None:
            return None
        try:
            return userdata.get(name)
        except Exception:
            return None

    kdir = Path.home() / ".kaggle"
    kdir.mkdir(exist_ok=True)

    token = secret("KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token.strip()
        f = kdir / "access_token"
        f.write_text(token.strip())
        f.chmod(0o600)
        return "KAGGLE_API_TOKEN"

    raw = secret("KAGGLE_JSON")
    if raw:
        f = kdir / "kaggle.json"
        f.write_text(raw if isinstance(raw, str) else json.dumps(raw))
        f.chmod(0o600)
        return "kaggle.json"

    user, key = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
    if user and key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
        return "KAGGLE_USERNAME + KAGGLE_KEY"

    for f in (kdir / "access_token", kdir / "kaggle.json"):
        if f.exists():
            return f"archivo existente {f.name}"

    raise RuntimeError(
        "No hay credenciales. Crea en Colab (icono de llave) el secreto KAGGLE_API_TOKEN "
        "con tu token KGAT_..., o KAGGLE_USERNAME + KAGGLE_KEY si usas el esquema antiguo."
    )

print("Credenciales via:", setup_kaggle_auth())

Credenciales via: KAGGLE_API_TOKEN


In [46]:
COMPETITION        = "biohub-cell-tracking-during-development"
DATA_DIR           = Path("/content/data")      # cambiar a Path("data") fuera de Colab
TRAIN_DIR          = DATA_DIR / "train"

# El listado de `train` solo expone 2 embriones, así que subir N_EMBRYOS no agrega
# nada: la palanca real para agrandar la muestra es SAMPLES_PER_EMBRYO.
# Costo: cada .geff pesa ~170 KB, pero explorar y bajar cada uno son ~20 llamadas
# a la API. Con 15 por embrión son ~30 muestras y unos 10-15 min de descarga.
N_EMBRYOS          = 8      # embriones distintos a analizar (tope, no garantía)
SAMPLES_PER_EMBRYO = 15     # muestras (fields of view) por embrión
IMAGE_BUDGET_MB    = 250    # tope de disco para el volumen de imagen que se descarga

# Escala física del vóxel (Z, Y, X) en micras/píxel, documentada por los organizadores.
VOXEL_UM = np.array([1.625, 0.40625, 0.40625])

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Destino:", DATA_DIR)

Destino: /content/data


In [47]:
from kaggle.api.kaggle_api_extended import KaggleApi
from kagglesdk.competitions.types.competition_api_service import ApiListDataTreeFilesRequest

api = KaggleApi()
api.authenticate()


def _is_fatal(err):
    # 403/404 no se reintenta: el archivo no existe o faltan permisos
    s = str(err)
    return "404" in s or "403" in s


def list_tree(path="", page_token=None, max_retries=6):
    # Lista una página de un directorio del dataset de la competencia
    for attempt in range(max_retries):
        try:
            with api.build_kaggle_client() as kaggle:
                req = ApiListDataTreeFilesRequest()
                req.competition_name = COMPETITION
                req.path = path
                if page_token:
                    req.page_token = page_token
                return kaggle.competitions.competition_api_client.list_data_tree_files(req)
        except Exception as e:
            if _is_fatal(e) or attempt == max_retries - 1:
                raise
            time.sleep(4 * (attempt + 1))


def list_tree_all(path=""):
    # Lista un directorio completo siguiendo la paginación
    dirs, files, token = [], [], None
    while True:
        r = list_tree(path, page_token=token)
        dirs += [d.name for d in getattr(r, "directories", [])]
        files += [f.name for f in getattr(r, "files", [])]
        token = getattr(r, "next_page_token", None)
        if not token:
            return dirs, files


def download_file(fname, max_retries=5):
    """Descarga un archivo suelto y lo deja en DATA_DIR/fname.

    ERROR ORIGINAL: se asumía que competition_download_file respeta la ruta
    completa. Dependiendo de la versión deja el archivo con otro nombre o
    comprimido, así que aquí se baja a un temporal y se coloca a mano."""
    target = DATA_DIR / fname
    if target.exists() and target.stat().st_size > 0:
        return target
    target.parent.mkdir(parents=True, exist_ok=True)

    for attempt in range(max_retries):
        tmp = Path(tempfile.mkdtemp())
        try:
            api.competition_download_file(COMPETITION, fname, path=str(tmp), quiet=True, force=True)
            got = [p for p in tmp.rglob("*") if p.is_file()]
            if not got:
                raise RuntimeError("descarga vacía")
            src = got[0]
            if src.suffix == ".zip" and not fname.endswith(".zip"):
                with zipfile.ZipFile(src) as z, open(target, "wb") as out:
                    with z.open(z.namelist()[0]) as member:
                        shutil.copyfileobj(member, out)
            else:
                shutil.move(str(src), str(target))
            return target
        except Exception as e:
            if _is_fatal(e) or attempt == max_retries - 1:
                raise
            time.sleep(4 * (attempt + 1))
        finally:
            shutil.rmtree(tmp, ignore_errors=True)


def download_many(fnames, workers=4, desc="Descargando"):
    ok, failed = [], []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(download_file, f): f for f in fnames}
        for fut in tqdm(as_completed(futs), total=len(futs), desc=desc):
            f = futs[fut]
            try:
                fut.result(); ok.append(f)
            except Exception as e:
                failed.append((f, str(e)[:120]))
    if failed:
        print(f"{len(failed)} archivos fallaron. Ejemplos:")
        for f, e in failed[:5]:
            print("  -", f, "->", e)
    return ok, failed

In [48]:
# --- Selección de muestras ---------------------------------------------------
# ERROR ORIGINAL: el bucle rompía al ver N_EMBRYOS embriones distintos, sin
# garantizar SAMPLES_PER_EMBRYO por embrión, y luego repetía el mismo cálculo dos
# veces. Además no verificaba que la muestra tuviera su .geff pareado.

zarr_stems, geff_stems = set(), set()
token, page = None, 0

while True:
    page += 1
    r = list_tree("train", page_token=token)
    for d in getattr(r, "directories", []):
        if d.name.endswith(".zarr"):
            zarr_stems.add(d.name[:-5])
        elif d.name.endswith(".geff"):
            geff_stems.add(d.name[:-5])

    paired = sorted(zarr_stems & geff_stems)
    by_embryo = defaultdict(list)
    for stem in paired:
        by_embryo[stem.split("_")[0]].append(stem)
    completos = [e for e, s in by_embryo.items() if len(s) >= SAMPLES_PER_EMBRYO]

    print(f"página {page}: {len(paired)} muestras pareadas, "
          f"{len(by_embryo)} embriones ({len(completos)} con >= {SAMPLES_PER_EMBRYO} muestras)")

    token = getattr(r, "next_page_token", None)
    if not token or len(completos) >= N_EMBRYOS:
        break
    time.sleep(1)

# Si no hay suficientes embriones completos, se completa con los que haya.
orden = sorted(by_embryo, key=lambda e: -len(by_embryo[e]))
SAMPLES = []
for embryo in orden[:N_EMBRYOS]:
    SAMPLES += sorted(by_embryo[embryo])[:SAMPLES_PER_EMBRYO]

print(f"\nEmbriones: {orden[:N_EMBRYOS]}")
print(f"Muestras seleccionadas ({len(SAMPLES)}):")
for s in SAMPLES:
    print("  -", s)

página 1: 10 muestras pareadas, 1 embriones (0 con >= 15 muestras)
página 2: 20 muestras pareadas, 1 embriones (1 con >= 15 muestras)
página 3: 30 muestras pareadas, 1 embriones (1 con >= 15 muestras)
página 4: 40 muestras pareadas, 1 embriones (1 con >= 15 muestras)
página 5: 50 muestras pareadas, 1 embriones (1 con >= 15 muestras)
página 6: 60 muestras pareadas, 1 embriones (1 con >= 15 muestras)
página 7: 70 muestras pareadas, 1 embriones (1 con >= 15 muestras)
página 8: 80 muestras pareadas, 2 embriones (1 con >= 15 muestras)
página 9: 90 muestras pareadas, 2 embriones (2 con >= 15 muestras)
página 10: 100 muestras pareadas, 2 embriones (2 con >= 15 muestras)
página 11: 110 muestras pareadas, 2 embriones (2 con >= 15 muestras)
página 12: 120 muestras pareadas, 2 embriones (2 con >= 15 muestras)
página 13: 130 muestras pareadas, 2 embriones (2 con >= 15 muestras)
página 14: 140 muestras pareadas, 2 embriones (2 con >= 15 muestras)
página 15: 150 muestras pareadas, 2 embriones (2 con

In [49]:
# --- Descarga de .geff sin listar el árbol de cada muestra --------------------
# El listado recursivo cuesta ~20 llamadas a la API por muestra y es lo que se
# colgaba. Aquí se explora UNA sola muestra para conocer la estructura, y para el
# resto se calculan los archivos exactos a partir de la metadata de cada arreglo.
import itertools

META_NAMES = {"zarr.json", ".zarray", ".zattrs", ".zgroup"}


def list_recursive(path):
    out, stack = [], [path]
    while stack:
        base = stack.pop()
        dirs, files = list_tree_all(base)
        out += [f"{base}/{f}" for f in files]
        stack += [f"{base}/{d}" for d in dirs]
    return out


def leer_meta(dir_local):
    # Metadata de un arreglo zarr; None si es un grupo o no existe
    p3, p2 = Path(dir_local) / "zarr.json", Path(dir_local) / ".zarray"
    if p3.exists():
        m = json.loads(p3.read_text())
        if m.get("node_type") == "group":
            return None
        return (tuple(m["shape"]), tuple(m["chunk_grid"]["configuration"]["chunk_shape"]),
                m.get("chunk_key_encoding", {"name": "default"}))
    if p2.exists():
        m = json.loads(p2.read_text())
        return (tuple(m["shape"]), tuple(m["chunks"]),
                {"name": "v2", "configuration": {"separator": "."}})
    return None


def chunks_de(dir_local, rel_dir):
    # Rutas de TODOS los chunks de un arreglo, derivadas de su metadata
    info = leer_meta(dir_local)
    if info is None:
        return []
    shape, ch, enc = info
    sep = enc.get("configuration", {}).get("separator",
                                           "." if enc.get("name") == "v2" else "/")
    pref = [] if enc.get("name") == "v2" else ["c"]
    grid = [range(-(-s // c)) for s, c in zip(shape, ch)]
    return [f"{rel_dir}/" + sep.join(pref + [str(i) for i in idx])
            for idx in itertools.product(*grid)]


def bajar_silencioso(files, workers=6):
    ok = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(download_file, f) for f in files]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="Descargando"):
            try:
                fut.result(); ok += 1
            except Exception:
                pass
    return ok


# 1) Estructura: se explora solo la primera muestra
t0 = time.time()
plantilla = [p.split(f"{SAMPLES[0]}.geff/", 1)[1]
             for p in list_recursive(f"train/{SAMPLES[0]}.geff")]
meta_rel = sorted({p for p in plantilla if Path(p).name in META_NAMES})
print(f"Estructura del .geff: {len(plantilla)} archivos, {len(meta_rel)} de metadata "
      f"({time.time()-t0:.0f}s)")

# 2) Metadata de las 30 muestras (misma estructura) + niveles del .zarr
metas = [f"train/{s}.geff/{r}" for s in SAMPLES for r in meta_rel]
metas += [f"train/{s}.zarr/{lvl}/zarr.json" for s in SAMPLES for lvl in ("0", "1", "2")]
print(f"\nBajando {len(metas)} archivos de metadata...")
bajar_silencioso(metas)

# 3) Chunks de datos, calculados sin llamar a la API
datos = []
for s in SAMPLES:
    for r in meta_rel:
        rel_dir = str(Path(r).parent)
        if rel_dir == ".":
            continue
        datos += chunks_de(TRAIN_DIR / f"{s}.geff" / rel_dir, f"train/{s}.geff/{rel_dir}")
datos = sorted(set(datos))
print(f"\nBajando {len(datos)} archivos de datos...")
bajar_silencioso(datos)

geff_ok = sum(1 for s in SAMPLES if (TRAIN_DIR / f"{s}.geff" / "nodes" / "ids").exists())
print(f"\nListo en {time.time()-t0:.0f}s | muestras con grafo completo: {geff_ok}/{len(SAMPLES)}")

Estructura del .geff: 21 archivos, 15 de metadata (17s)

Bajando 540 archivos de metadata...


Descargando:   0%|          | 0/540 [00:00<?, ?it/s]


Bajando 180 archivos de datos...


Descargando:   0%|          | 0/180 [00:00<?, ?it/s]


Listo en 51s | muestras con grafo completo: 30/30


In [50]:
def zarr_json(path):
    """Lee la metadata de un arreglo zarr (v3: zarr.json, v2: .zarray)."""
    p3, p2 = Path(path) / "zarr.json", Path(path) / ".zarray"
    if p3.exists():
        m = json.loads(p3.read_text())
        if m.get("node_type") == "group":
            return None
        return {"shape": tuple(m["shape"]),
                "dtype": str(m.get("data_type", m.get("dtype"))),
                "chunks": tuple(m["chunk_grid"]["configuration"]["chunk_shape"]),
                "encoding": m.get("chunk_key_encoding", {"name": "default"}),
                "raw": m}
    if p2.exists():
        m = json.loads(p2.read_text())
        return {"shape": tuple(m["shape"]), "dtype": str(m["dtype"]),
                "chunks": tuple(m["chunks"]),
                "encoding": {"name": "v2", "configuration": {"separator": "."}},
                "raw": m}
    return None


def image_levels(stem):
    """Niveles de la pirámide multiescala disponibles localmente, con su metadata."""
    root = TRAIN_DIR / f"{stem}.zarr"
    out = {}
    if not root.exists():
        return out
    for d in sorted(root.iterdir()):
        if d.is_dir():
            m = zarr_json(d)
            if m:
                out[d.name] = m
    return out


def read_geff(stem):
    """Lee un grafo .geff -> (atributos, ids de nodo, props dict, aristas)."""
    g = TRAIN_DIR / f"{stem}.geff"
    attrs = {}
    try:
        attrs = dict(zarr.open_group(str(g), mode="r").attrs)
    except Exception:
        pass

    node_ids = np.asarray(zarr.open(str(g / "nodes" / "ids"), mode="r")[:])
    props = {}
    for name in ("t", "z", "y", "x"):
        p = g / "nodes" / "props" / name / "values"
        props[name] = (np.asarray(zarr.open(str(p), mode="r")[:]).astype(float)
                       if p.exists() else np.full(node_ids.shape[0], np.nan))

    ep = g / "edges" / "ids"
    edges = np.asarray(zarr.open(str(ep), mode="r")[:]) if ep.exists() else np.zeros((0, 2), int)
    return attrs, node_ids, props, edges


def find_key(d, key):
    """Busca una llave recursivamente en atributos anidados."""
    if not isinstance(d, dict):
        return None
    if key in d:
        return d[key]
    for v in d.values():
        r = find_key(v, key)
        if r is not None:
            return r
    return None

In [51]:
# --- Diagnóstico y reparación de descargas incompletas -----------------------
# bajar_silencioso ignora los fallos, así que puede quedar un directorio creado
# pero vacío. Aquí se detecta qué falta y se reintenta mostrando los errores.

def tiene_meta(d):
    return (Path(d) / "zarr.json").exists() or (Path(d) / ".zarray").exists()


ARRAYS_REQ = ["nodes/ids", "edges/ids"] + [f"nodes/props/{n}/values" for n in ("t", "z", "y", "x")]

faltantes = {}
for s in SAMPLES:
    base = TRAIN_DIR / f"{s}.geff"
    falta = [r for r in ARRAYS_REQ if not tiene_meta(base / r)]
    if falta:
        faltantes[s] = falta

print(f"Muestras con arreglos incompletos: {len(faltantes)}/{len(SAMPLES)}")
for s, f in list(faltantes.items())[:10]:
    print(f"  {s}: falta {f}")

# Reintento visible de la metadata que falta
if faltantes:
    pendientes = [f"train/{s}.geff/{r}/zarr.json" for s, rs in faltantes.items() for r in rs]
    print(f"\nReintentando {len(pendientes)} archivos de metadata...")
    ok, fallidos = download_many(pendientes, workers=4, desc="Metadata faltante")

    # Y los chunks de los arreglos que sí se recuperaron
    nuevos = []
    for s, rs in faltantes.items():
        for r in rs:
            nuevos += chunks_de(TRAIN_DIR / f"{s}.geff" / r, f"train/{s}.geff/{r}")
    if nuevos:
        print(f"\nBajando {len(nuevos)} chunks de datos...")
        bajar_silencioso(nuevos)

# Verificación final
INCOMPLETAS = [s for s in SAMPLES
               if not all(tiene_meta(TRAIN_DIR / f"{s}.geff" / r) for r in ARRAYS_REQ)]
SAMPLES_OK = [s for s in SAMPLES if s not in INCOMPLETAS]
print(f"\nMuestras utilizables: {len(SAMPLES_OK)}/{len(SAMPLES)}")
if INCOMPLETAS:
    print("Se excluyen del análisis:", INCOMPLETAS)

Muestras con arreglos incompletos: 0/30

Muestras utilizables: 30/30


In [52]:
# --- read_geff tolerante -----------------------------------------------------
# Antes comprobaba que el directorio existiera; ahora exige que tenga metadata,
# porque una descarga fallida deja la carpeta creada pero vacía.

def read_geff(stem):
    g = TRAIN_DIR / f"{stem}.geff"
    attrs = {}
    try:
        attrs = dict(zarr.open_group(str(g), mode="r").attrs)
    except Exception:
        pass

    node_ids = np.asarray(zarr.open(str(g / "nodes" / "ids"), mode="r")[:])
    props = {}
    for name in ("t", "z", "y", "x"):
        p = g / "nodes" / "props" / name / "values"
        if tiene_meta(p):
            props[name] = np.asarray(zarr.open(str(p), mode="r")[:]).astype(float)
        else:
            print(f"AVISO: {stem} no tiene la propiedad '{name}'")
            props[name] = np.full(node_ids.shape[0], np.nan)

    ep = g / "edges" / "ids"
    edges = np.asarray(zarr.open(str(ep), mode="r")[:]) if tiene_meta(ep) else np.zeros((0, 2), int)
    return attrs, node_ids, props, edges


# A partir de aquí el análisis usa solo las muestras completas
SAMPLES = SAMPLES_OK
print(f"SAMPLES = {len(SAMPLES)} muestras completas")

SAMPLES = 30 muestras completas


In [53]:
# --- Recuperación paciente: secuencial, con reintento de los 404 -------------
import random

def bajar_terco(fname, intentos=6):
    target = DATA_DIR / fname
    if target.exists() and target.stat().st_size > 0:
        return True
    target.parent.mkdir(parents=True, exist_ok=True)
    for k in range(intentos):
        tmp = Path(tempfile.mkdtemp())
        try:
            api.competition_download_file(COMPETITION, fname, path=str(tmp), quiet=True, force=True)
            got = [p for p in tmp.rglob("*") if p.is_file()]
            if got:
                shutil.move(str(got[0]), str(target))
                return True
        except Exception:
            pass
        finally:
            shutil.rmtree(tmp, ignore_errors=True)
        time.sleep(min(30, 3 * 2 ** k) + random.uniform(0, 2))   # 3, 6, 12, 24, 30, 30 s
    return False


pendientes = sorted(set(restantes))
print(f"Pendientes: {len(pendientes)} | estimado: {len(pendientes) * 4 / 60:.0f}-{len(pendientes) * 15 / 60:.0f} min\n")

recuperados, sin_suerte = 0, []
for k, f in enumerate(tqdm(pendientes, desc="Recuperando"), 1):
    if bajar_terco(f):
        recuperados += 1
    else:
        sin_suerte.append(f)
    time.sleep(1.5)                       # espaciado entre archivos
    if k % 20 == 0:
        print(f"  {k}/{len(pendientes)}: {recuperados} ok, {len(sin_suerte)} sin suerte", flush=True)

print(f"\nRecuperados: {recuperados} | siguen faltando: {len(sin_suerte)}")

Pendientes: 102 | estimado: 7-26 min



Recuperando:   0%|          | 0/102 [00:00<?, ?it/s]

  20/102: 20 ok, 0 sin suerte
  40/102: 40 ok, 0 sin suerte
  60/102: 60 ok, 0 sin suerte
  80/102: 80 ok, 0 sin suerte
  100/102: 100 ok, 0 sin suerte

Recuperados: 102 | siguen faltando: 0


In [54]:
# --- Filtro estricto: solo muestras con TODOS los chunks y datos coherentes ---
def muestra_valida(s):
    for r in ARRAYS_REQ:
        rutas = chunks_de(TRAIN_DIR / f"{s}.geff" / r, f"train/{s}.geff/{r}")
        if not rutas or any(not (DATA_DIR / x).exists() for x in rutas):
            return False, f"faltan chunks en {r}"
    try:
        _, ids, props, edges = read_geff(s)
        if len(np.unique(ids)) < len(ids) * 0.9:
            return False, "ids repetidos (datos en cero)"
        if np.nanmax(props["t"]) == np.nanmin(props["t"]):
            return False, "t constante"
    except Exception as e:
        return False, str(e)[:50]
    return True, "ok"


validas, descartadas = [], {}
for s in SAMPLES_OK if "SAMPLES_OK" in dir() else SAMPLES:
    ok, razon = muestra_valida(s)
    (validas.append(s) if ok else descartadas.update({s: razon}))

print(f"Muestras válidas: {len(validas)}")
for s, r in list(descartadas.items())[:10]:
    print(f"  descartada {s}: {r}")

SAMPLES = validas
assert SAMPLES, "No quedó ninguna muestra completa."
print(f"\nSAMPLES = {len(SAMPLES)} muestras | embriones: "
      f"{sorted({s.split('_')[0] for s in SAMPLES})}")

Muestras válidas: 30

SAMPLES = 30 muestras | embriones: ['44b6', '6bba']


In [55]:
# --- Construcción de las tres tablas ----------------------------------------
node_rows, edge_rows, sample_rows = [], [], []

for stem in tqdm(SAMPLES, desc="Construyendo tablas"):
    embryo = stem.split("_")[0]
    attrs, ids, props, edges = read_geff(stem)

    n = len(ids)
    idx = {int(v): i for i, v in enumerate(ids)}          # id de nodo -> posición
    indeg, outdeg = np.zeros(n, int), np.zeros(n, int)

    e_src = np.array([idx[int(a)] for a, b in edges if int(a) in idx and int(b) in idx], int)
    e_dst = np.array([idx[int(b)] for a, b in edges if int(a) in idx and int(b) in idx], int)
    for i in e_src:
        outdeg[i] += 1
    for i in e_dst:
        indeg[i] += 1

    nodes = pd.DataFrame({
        "sample": stem, "embryo_id": embryo, "node_id": ids,
        "t": props["t"], "z": props["z"], "y": props["y"], "x": props["x"],
        "in_degree": indeg, "out_degree": outdeg,
    })
    nodes["z_um"] = nodes["z"] * VOXEL_UM[0]
    nodes["y_um"] = nodes["y"] * VOXEL_UM[1]
    nodes["x_um"] = nodes["x"] * VOXEL_UM[2]
    nodes["es_division"] = nodes["out_degree"] >= 2
    nodes["tipo_temporal"] = np.select(
        [(nodes.in_degree == 0) & (nodes.out_degree == 0),
         nodes.in_degree == 0,
         nodes.out_degree == 0],
        ["aislado", "inicio", "fin"], default="intermedio")
    node_rows.append(nodes)

    if len(e_src):
        a, b = nodes.iloc[e_src].reset_index(drop=True), nodes.iloc[e_dst].reset_index(drop=True)
        d = np.stack([(b.z_um - a.z_um), (b.y_um - a.y_um), (b.x_um - a.x_um)], 1)
        edge_rows.append(pd.DataFrame({
            "sample": stem, "embryo_id": embryo,
            "t_origen": a.t.values, "dt": (b.t - a.t).values,
            "dz_um": d[:, 0], "dy_um": d[:, 1], "dx_um": d[:, 2],
            "dist_um": np.linalg.norm(d, axis=1),
            "out_degree_origen": a.out_degree.values,
        }))

    lvls = image_levels(stem)
    lvl0 = lvls.get("0") or (list(lvls.values())[0] if lvls else None)
    shape0 = lvl0["shape"] if lvl0 else (np.nan,) * 4

    sample_rows.append({
        "sample": stem, "embryo_id": embryo, "field_of_view": "_".join(stem.split("_")[1:]),
        "T": shape0[0], "Z": shape0[1], "Y": shape0[2], "X": shape0[3],
        "dtype": lvl0["dtype"] if lvl0 else None,
        "n_niveles_piramide": len(lvls),
        "n_nodos": n, "n_aristas": len(e_src),
        "t_min": np.nanmin(props["t"]) if n else np.nan,
        "t_max": np.nanmax(props["t"]) if n else np.nan,
        "n_t_anotados": len(np.unique(props["t"])) if n else 0,
        "n_divisiones": int((outdeg >= 2).sum()),
        "estimated_number_of_nodes": find_key(attrs, "estimated_number_of_nodes"),
    })

EDGE_COLS = ["sample", "embryo_id", "t_origen", "dt", "dz_um", "dy_um", "dx_um",
             "dist_um", "out_degree_origen"]
df_nodes = pd.concat(node_rows, ignore_index=True)
df_edges = (pd.concat(edge_rows, ignore_index=True) if edge_rows
            else pd.DataFrame(columns=EDGE_COLS).astype(float))
df_samples = pd.DataFrame(sample_rows)

# Variables derivadas a nivel de muestra
df_samples["cobertura_temporal"] = df_samples["n_t_anotados"] / df_samples["T"]
df_samples["nodos_por_t"] = df_samples["n_nodos"] / df_samples["n_t_anotados"]
df_samples["aristas_por_nodo"] = df_samples["n_aristas"] / df_samples["n_nodos"]
df_samples["tasa_division"] = df_samples["n_divisiones"] / df_samples["n_nodos"]

print(f"df_samples: {df_samples.shape} | df_nodes: {df_nodes.shape} | df_edges: {df_edges.shape}")
df_samples

Construyendo tablas:   0%|          | 0/30 [00:00<?, ?it/s]

df_samples: (30, 20) | df_nodes: (15934, 14) | df_edges: (15369, 9)


,sample,embryo_id,field_of_view,T,Z,Y,X,dtype,n_niveles_piramide,n_nodos,n_aristas,t_min,t_max,n_t_anotados,n_divisiones,estimated_number_of_nodes,cobertura_temporal,nodos_por_t,aristas_por_nodo,tasa_division
0,6bba_05b6850b,6bba,05b6850b,100,64,256,256,uint16,1,861,845,0.0,99.0,100,0,6362,1.00,8.610000,0.981417,0.000000
1,6bba_05db0fb1,6bba,05db0fb1,100,64,256,256,uint16,1,1229,1183,0.0,99.0,100,3,69800,1.00,12.290000,0.962571,0.002441
2,6bba_062c8d37,6bba,062c8d37,100,64,256,256,uint16,1,930,898,0.0,99.0,99,1,6030,0.99,9.393939,0.965591,0.001075
3,6bba_07477033,6bba,07477033,100,64,256,256,uint16,1,613,602,0.0,99.0,100,0,4836,1.00,6.130000,0.982055,0.000000
4,6bba_07e24132,6bba,07e24132,100,64,256,256,uint16,1,359,345,0.0,99.0,100,2,21485,1.00,3.590000,0.961003,0.005571
5,6bba_085bf656,6bba,085bf656,100,64,256,256,uint16,1,1198,1168,0.0,99.0,100,1,8463,1.00,11.980000,0.974958,0.000835
6,6bba_09961292,6bba,09961292,100,64,256,256,uint16,1,1950,1871,0.0,99.0,100,4,31117,1.00,19.500000,0.959487,0.002051
7,6bba_0c7fa718,6bba,0c7fa718,100,64,256,256,uint16,1,704,659,0.0,99.0,94,0,13650,0.94,7.489362,0.936080,0.000000
8,6bba_0e7c0d07,6bba,0e7c0d07,100,64,256,256,uint16,1,209,198,0.0,85.0,84,1,22467,0.84,2.488095,0.947368,0.004785
9,6bba_12665c0e,6bba,12665c0e,100,64,256,256,uint16,1,1036,998,0.0,99.0,100,3,8314,1.00,10.360000,0.963320,0.002896
